# 长鑫科技合理估值分析（二）数据清洗与统计事实

本 Notebook 完成：
- 数据清洗与指标构造（第 3 节）
- 描述性统计、时间趋势、分组对比、事件研究（第 4 节）


## 3. 数据来源与处理

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA, CLEAN, OUTPUT = ROOT / "data", ROOT / "clean", ROOT / "output"
CLEAN.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)

STOCK_LIST = [
    {"code": "603986", "name": "兆易创新", "group": "存储产业链"},
    {"code": "688008", "name": "澜起科技", "group": "存储产业链"},
    {"code": "300223", "name": "北京君正", "group": "存储产业链"},
    {"code": "688981", "name": "中芯国际", "group": "制造"},
    {"code": "002371", "name": "北方华创", "group": "设备"},
    {"code": "688012", "name": "中微公司", "group": "设备"},
    {"code": "688396", "name": "华润微", "group": "制造"},
    {"code": "603501", "name": "韦尔股份", "group": "设计"},
]

import os
import sys
os.chdir(ROOT)  # 加载 DS/matplotlibrc，避免中文乱码
sys.path.insert(0, str(ROOT / "scripts"))
from plot_style import setup_theme, save_figure, create_figure, set_title, add_source, PALETTE
setup_theme()  # 必须在 seaborn 样式之后覆盖中文字体
print("路径 OK:", DATA.exists(), CLEAN.exists())


### 3.1 清洗步骤

1. **行情**：合并 8 家可比公司后复权日线，计算日收益率 `return`，保存 `clean/stock_clean.csv`。
2. **招股书财务**：去重、换算亿元 `value_bn`，宽表 `clean/cxkj_financials_wide.csv`。
3. **构造指标**：营收同比增速、研发费用率、毛利率、同业 ROE 面板。


In [ ]:
# --- 清洗：A 股行情 ---
frames = []
for item in STOCK_LIST:
    df = pd.read_csv(DATA / "stock" / f"stock_{item['code']}.csv", parse_dates=["date"])
    df["code"], df["name"], df["group"] = item["code"], item["name"], item["group"]
    df["return"] = df["close"].pct_change()
    frames.append(df)
stock = pd.concat(frames, ignore_index=True)
stock.to_csv(CLEAN / "stock_clean.csv", index=False, encoding="utf-8-sig")

# --- 清洗：长鑫财务 ---
fin = pd.read_csv(DATA / "ipo" / "cxkj_financials.csv")
fin = fin.drop_duplicates(subset=["period", "metric"], keep="first")
fin["value_bn"] = fin["value"] / 1e8
fin.to_csv(CLEAN / "cxkj_financials_clean.csv", index=False, encoding="utf-8-sig")
wide = fin.pivot_table(index="period", columns="metric", values="value_bn", aggfunc="first")
wide.to_csv(CLEAN / "cxkj_financials_wide.csv", encoding="utf-8-sig")

# --- 构造指标 ---
rev = fin[fin["metric"] == "revenue"].copy()
annual = rev[rev["period"].isin(["2022", "2023", "2024"])].sort_values("period")
annual["yoy_pct"] = annual["value_bn"].pct_change() * 100
cost = fin[fin["metric"] == "cost"].set_index("period")["value_bn"]
rd = fin[fin["metric"] == "rd_expense"].set_index("period")["value_bn"]
metrics = annual.set_index("period")
metrics["gross_margin_pct"] = (metrics["value_bn"] - cost.reindex(metrics.index)) / metrics["value_bn"] * 100
metrics["rd_ratio_pct"] = rd.reindex(metrics.index) / metrics["value_bn"] * 100
metrics = metrics.reset_index()
metrics.to_csv(CLEAN / "cxkj_derived_metrics.csv", index=False, encoding="utf-8-sig")
display(metrics[["period", "value_bn", "yoy_pct", "gross_margin_pct", "rd_ratio_pct"]])


## 4. 统计事实

### 4.1 描述性统计：长鑫规模与盈利

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
rev_plot = fin[fin["metric"] == "revenue"].copy()
rev_plot["period"] = pd.Categorical(rev_plot["period"], ["2022", "2023", "2024", "2025H1"], ordered=True)
rev_plot = rev_plot.sort_values("period")
np_ = fin[fin["metric"] == "net_profit_parent"]
ax1.bar(rev_plot["period"].astype(str), rev_plot["value_bn"], color="#4a90e2", alpha=0.85, label="营业收入")
ax1.set_ylabel("营业收入（亿元）")
ax1.set_title("长鑫科技营业收入与归母净利润")
ax2 = ax1.twinx()
ax2.plot(np_["period"].astype(str), np_["value_bn"], "o-", color="#e74c3c", label="归母净利润")
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_ylabel("归母净利润（亿元）")
ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUTPUT / "fig1_cxkj_revenue_profit.png", dpi=150)
plt.show()


**解读**：2023—2024 年营收从约 91 亿元跃升至 242 亿元，增速约 166%，反映 DRAM 景气上行与产能释放；归母净利润仍为负且绝对值较大，显示**高成长与高亏损并存**的典型早期晶圆厂特征。2025 年上半年营收 154 亿元，年化收入规模继续抬升，但盈利拐点尚未在报表确认。


### 4.2 产品结构

In [ ]:
sub = fin[fin["metric"].str.startswith("revenue_") & ~fin["metric"].str.contains("小计")].copy()
sub["product"] = sub["metric"].str.replace("revenue_", "", regex=False)
wide_p = sub.pivot(index="period", columns="product", values="value_bn")
wide_p.plot(kind="bar", figsize=(8, 5), color=["#667eea", "#f6ad55"])
plt.title("分产品收入（亿元）"); plt.ylabel("收入"); plt.legend(title="产品线")
plt.tight_layout()
plt.savefig(OUTPUT / "fig2_product_mix.png", dpi=150)
plt.show()
display(wide_p)


**解读**：LPDDR 系列收入占比显著高于 DDR，2025H1 LPDDR 约 106 亿元、DDR 约 42 亿元，说明公司产品重心在移动端/低功耗场景。结构变化将影响毛利率与资本开支节奏，估值上宜对标**移动 DRAM 占比更高**的同业或周期阶段。


### 4.3 时间趋势：可比公司股价（2020=1）

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
base_date = pd.Timestamp("2020-01-02")
for name, g in stock.groupby("name"):
    g = g.sort_values("date")
    base = g.loc[g["date"] >= base_date, "close"].iloc[0]
    ax.plot(g["date"], g["close"] / base, label=name, linewidth=1.1)
ax.set_title("可比公司股价（后复权，2020-01-02=1）")
ax.set_ylabel("归一化价格"); ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT / "fig3_peer_normalized.png", dpi=150)
plt.show()


**解读**：存储产业链（兆易创新、澜起科技、北京君正）与设备龙头（北方华创）走势分化明显，2023—2024 年存储相关标的波动加剧，与全球 DRAM 价格周期共振。长鑫未上市，其合理估值应参考**周期高点/低点**下同业市值—收入倍数区间，而非单一时点股价。


### 4.4 分组对比：产业链 vs 设备 ROE

In [ ]:
def parse_ths_number(x):
    if pd.isna(x):
        return None
    s = str(x).strip().replace(",", "")
    if s.endswith("%"):
        return float(s[:-1])
    try:
        return float(s)
    except ValueError:
        return None

roe_rows = []
for item in STOCK_LIST:
    p = DATA / "finance" / f"finance_ths_{item['code']}.csv"
    if not p.exists():
        continue
    df = pd.read_csv(p)
    col_roe = [c for c in df.columns if "净资产收益率" in c][0]
    col_period = df.columns[0]
    sub = df[[col_period, col_roe]].copy()
    sub["year"] = pd.to_datetime(sub[col_period], errors="coerce").dt.year
    sub["roe"] = sub[col_roe].map(parse_ths_number)
    sub = sub.dropna().groupby("year", as_index=False).last()
    for _, r in sub.iterrows():
        if r["year"] >= 2020:
            roe_rows.append({"name": item["name"], "group": item["group"], "year": int(r["year"]), "roe": r["roe"]})
roe_df = pd.DataFrame(roe_rows)
roe_df.to_csv(CLEAN / "peer_roe_clean.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(10, 5))
for name, g in roe_df.groupby("name"):
    ax.plot(g["year"], g["roe"], marker="o", label=name)
ax.axhline(0, color="gray", linestyle="--")
ax.set_title("可比公司 ROE (%)"); ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT / "fig5_roe_peer.png", dpi=150)
plt.show()

print("按产业链分组 2024 年 ROE 均值：")
print(roe_df[roe_df["year"] == 2024].groupby("group")["roe"].mean().round(2))


**解读**：设备与制造龙头 ROE 在 2024 年多为正且高于部分存储设计公司；长鑫尚未盈利，不能直接套用同业 PE，但**盈利转正后的 ROE 路径**是决定长期估值中枢的关键假设。


### 4.5 政策/事件前后：半导体指数累计收益

In [ ]:
import matplotlib.dates as mdates

semi = pd.read_csv(DATA / "industry" / "semiconductor_index_ths.csv", parse_dates=["日期"])
semi = semi.rename(columns={"日期": "date", "收盘价": "close"}).sort_values("date").set_index("date")
semi["ret"] = semi["close"].pct_change()

events = [
    {"date": "2025-07-07", "event": "IPO辅导备案"},
    {"date": "2025-12-30", "event": "招股说明书披露"},
]
window = 20
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, ev in zip(axes, events):
    t0 = pd.Timestamp(ev["date"])
    w = semi.loc[(semi.index >= t0 - pd.Timedelta(days=window)) & (semi.index <= t0 + pd.Timedelta(days=window))].copy()
    w["rel_day"] = (w.index - t0).days
    w["car"] = (1 + w["ret"]).cumprod() - 1
    ax.plot(w["rel_day"], w["car"] * 100, color="#4a90e2")
    ax.axvline(0, color="#e74c3c", linestyle="--")
    ax.set_title(ev["event"], fontsize=9)
    ax.set_xlabel("相对交易日"); ax.set_ylabel("累计收益 (%)")
plt.suptitle("IPO 节点前后半导体指数 CAR（事件研究示意）")
plt.tight_layout()
plt.savefig(OUTPUT / "fig7_event_study.png", dpi=150)
plt.show()


**解读**：事件窗口内半导体指数累计收益波动有限，说明长鑫 IPO 消息对板块整体**短期冲击温和**，更多体现为主题情绪而非系统性重估。需更长样本与多事件叠加检验显著性（见 Notebook 3）。


### 4.6 全球 DRAM 龙头 vs A 股半导体

In [ ]:
us = pd.read_csv(CLEAN / "us_clean.csv", parse_dates=["date"]) if (CLEAN / "us_clean.csv").exists() else None
if us is None:
    uframes = []
    for sym, nm in [("MU", "美光科技"), ("WDC", "西部数据")]:
        u = pd.read_csv(DATA / "us" / f"us_{sym}.csv", parse_dates=["date"])
        u["symbol"], u["name"] = sym, nm
        uframes.append(u)
    us = pd.concat(uframes)

mu = us[us["symbol"] == "MU"].set_index("date")["close"]
mu_n = mu / mu.iloc[0]
semi_n = semi["close"] / semi["close"].iloc[0]
common = mu_n.index.intersection(semi_n.index)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(common, mu_n.loc[common], label="美光 MU", color="#2d6a4f")
ax.plot(common, semi_n.loc[common], label="A股半导体指数", color="#9b5de5")
ax.set_title("美光 vs 半导体指数（2020=1）"); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "fig4_mu_vs_semiconductor.png", dpi=150)
plt.show()


**解读**：美光股价与 A 股半导体指数相关但不完全一致，2022—2023 年 MU 大幅回撤阶段对应全球存储去库存。长鑫估值需嵌入**全球 DRAM 价格周期**，一级市场 1282—1584 亿元报道区间与周期位置高度相关。
